In [1]:
!pip install ai2-olmo

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.9/144.9 MB 11.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.8/56.8 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 242.4/242.4 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.6 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.4 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 4.9 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 31.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 13.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 2.1 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 85.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 160.8/160.8 kB 9.3 MB/s eta 0:00:00
  Attempting uninstall: nvidi

In [ ]:
import os
import json
import numpy as np
from datasets import load_dataset
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from tqdm import tqdm
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, AutoConfig


class OmniDNAEmbeddingExtractor:
    """
    Класс для извлечения эмбеддингов последовательностей ДНК/РНК
    из модели Omni-DNA.
    """
    
    def __init__(self, model_name: str = "zehui127/Omni-DNA-300M", device: torch.device = None):
        self.device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")
        config = AutoConfig.from_pretrained(model_name, trust_remote_code=True, output_hidden_states=True)
        self.tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
        self.model = AutoModelForCausalLM.from_pretrained(model_name, config=config, trust_remote_code=True)
        self.model.to(self.device).eval()

    def extract_embeddings(
        self,
        sequences: list[str],
        batch_size: int = 8,
        max_length: int = 512
    ) -> np.ndarray:
        """
        Извлекает эмбеддинги для списка последовательностей методом mean pooling
        по последнему hidden state модели.

        Args:
            sequences: список строк ДНК/РНК
            batch_size: размер батча для инференса
            max_length: максимальная длина токенов (усечение)

        Returns:
            np.ndarray формы (N, H) — эмбеддинги
        """
        all_embeds = []
        with torch.no_grad():
            for i in tqdm(range(0, len(sequences), batch_size), desc="Extract embeddings"):
                batch = sequences[i : i + batch_size]
                enc = self.tokenizer(
                    batch,
                    return_tensors="pt",
                    padding=True,
                    truncation=True,
                    max_length=max_length
                ).to(self.device)

                # Генерируем выдачу с hidden_states
                outputs = self.model(**enc)
                # Последний слой скрытых состояний
                hidden_states = outputs.hidden_states[-1]  # (B, L, H)

                # mean pooling по токенам (с учётом маски)
                mask = enc.attention_mask.unsqueeze(-1)
                summed = (hidden_states * mask).sum(dim=1)
                counts = mask.sum(dim=1).clamp(min=1)
                embeds = (summed / counts).cpu().numpy()

                all_embeds.append(embeds)

        return np.vstack(all_embeds)


# Few-shot эксперименты
def few_shot(train, test, model_name, ks=(1, 5, 10, 20), trials=5):
    res = {}
    rng = np.random.RandomState(42)
    for task in tqdm(set(train['task']), desc='Few-shot'):
        tr = train.filter(lambda x, t=task: x['task']==t)
        te = test.filter(lambda x, t=task: x['task']==t)
        seqs_tr, y_tr = tr['sequence'], np.array(tr['label'])
        seqs_te, y_te = te['sequence'], np.array(te['label'])
        X_te = extractor.extract_embeddings(seqs_te, batch_size=BATCH_SIZE)
        res[task] = {}

        for k in ks:
            accs, f1s = [], []
            for _ in range(trials):
                idxs = []
                for lbl in np.unique(y_tr):
                    locs = np.where(y_tr == lbl)[0]
                    choice = rng.choice(locs, size=min(k, len(locs)), replace=False)
                    idxs.extend(choice.tolist())

                X_k = extractor.extract_embeddings([seqs_tr[i] for i in idxs], batch_size=BATCH_SIZE)
                y_k = y_tr[idxs]

                clf = LogisticRegression(**params_logreg)
                Xf = X_k.reshape(-1, 1) if X_k.ndim == 1 or X_k.shape[1] == 1 else X_k
                Xt = X_te.reshape(-1, 1) if X_te.ndim == 1 or X_te.shape[1] == 1 else X_te
                clf.fit(Xf, y_k)
                p = clf.predict(Xt)
                accs.append(accuracy_score(y_te, p))
                f1s.append(f1_score(y_te, p, average='macro'))

            res[task][k] = {
                'accuracy': float(np.mean(accs)),
                'f1_score': float(np.mean(f1s))
            }
            with open(f'{PATH_TO_SAVE_OUTPUTS}/results_{model_name.split("/")[1].lower()}_task-{task}_k-{k}.json', 'w') as f:
                json.dump(res, f, indent=4)
    return res


ds = load_dataset("InstaDeepAI/nucleotide_transformer_downstream_tasks", trust_remote_code=True)
train_ds, test_ds = ds['train'], ds['test']
models = [
        'zehui127/Omni-DNA-20M',
        'zehui127/Omni-DNA-60M',
        'zehui127/Omni-DNA-116M',
        'zehui127/Omni-DNA-300M',
        'zehui127/Omni-DNA-700M',
        'zehui127/Omni-DNA-1B' 
    ]

PATH_TO_SAVE_OUTPUTS = '/kaggle/working'
PARAMS_LOGREG = {'max_iter': 1000, 'random_state': 42}
BATCH_SIZE = 16

# Baseline: обучение на полном наборе
for model_name in models:

    extractor = OmniDNAEmbeddingExtractor(model_name=model_name)
    
    baseline = {}
    for task in tqdm(set(train_ds['task']), desc='Baseline'):
        tr = train_ds.filter(lambda x, t=task: x['task']==t)
        te = test_ds.filter(lambda x, t=task: x['task']==t)
        seqs_tr, y_tr = tr['sequence'], np.array(tr['label'])
        seqs_te, y_te = te['sequence'], np.array(te['label'])
    
        X_tr = extractor.extract_embeddings(seqs_tr, batch_size=BATCH_SIZE)
        X_te = extractor.extract_embeddings(seqs_te, batch_size=BATCH_SIZE)
    
        clf = LogisticRegression(**PARAMS_LOGREG)
        Xf = X_tr.reshape(-1, 1) if X_tr.ndim == 1 or X_tr.shape[1] == 1 else X_tr
        Xt = X_te.reshape(-1, 1) if X_te.ndim == 1 or X_te.shape[1] == 1 else X_te
        clf.fit(Xf, y_tr)
        preds = clf.predict(Xt)
    
        baseline[task] = {
            'accuracy': float(accuracy_score(y_te, preds)),
            'f1_score': float(f1_score(y_te, preds, average='macro'))
        }
        with open(f'{PATH_TO_SAVE_OUTPUTS}/results_{model_name.split("/")[1].lower()}_task-{task}_baseline.json', 'w') as f:
            json.dump(baseline, f, indent=4)
    
    results_kshot = few_shot(train_ds, test_ds, model_name)
    
    output = {'full': baseline, 'kshot': results_kshot, 'params': PARAMS_LOGREG}
    with open(f'{PATH_TO_SAVE_OUTPUTS}/results_{model_name.split("/")[1].lower()}.json', 'w') as f:
        json.dump(output, f, indent=4)



Extract embeddings: 100%|██████████| 1859/1859 [00:53<00:00, 34.87it/s]

Extract embeddings: 100%|██████████| 207/207 [00:05<00:00, 34.90it/s]
/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_logistic.py:458: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
Extract embeddings:  66%|██████▋   | 1372/2070 [00:39<00:19, 35.06it/s]